In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import mean_absolute_error
import os
import joblib
import numpy as np
from typing import Tuple

In [2]:
df = pd.read_csv(
    "data/learning_traces.13m.csv",
    engine="pyarrow",
    dtype_backend="pyarrow",
    dtype={"lexeme_string": "string[pyarrow]"},
)

In [3]:
df.sample(n=8)

,p_recall,timestamp,delta,user_id,learning_language,ui_language,lexeme_id,lexeme_string,history_seen,history_correct,session_seen,session_correct
4640228,1.0,1362448474,345845,u:bpBB,en,es,450e428ac43a7205f1dca1134aa702fa,all/all<predet><sp>,17,16,1,1
5386212,1.0,1362511953,238,u:haSi,en,pt,e35a9db5692415795f36448f6dff95d2,talk/talk<vblex><pres>,6,6,1,1
11541384,1.0,1363007069,1062,u:hkKT,en,es,efc2c65bd3a2ab7e0e2d4db65e290699,read/read<vblex><pres>,18,18,4,4
6082613,1.0,1362562874,158,u:gVcq,es,en,a18e94e6d95abfb6d23e941ef0132505,adiós/adiós<ij>,8,8,3,3
6126385,1.0,1362568618,1043,u:i160,es,en,fda174b61b0ed5e2eaa6deceae37f56b,son/ser<vbser><pri><p3><pl>,6,4,1,1
5227812,1.0,1362503084,1682,u:hBni,en,es,800743ae07452296ded9823d6f1e2664,design/design<vblex><pres>,3,3,1,1
8188109,1.0,1362709869,279,u:i1OZ,fr,en,5c7ea67876ecc95f79b3ceaf699cd7d2,boire/boire<vblex><inf>,5,4,1,1
12073255,1.0,1363039232,801454,u:iKkp,es,en,61c2f4dd9c024c9d99910ba2b04590b1,comen/comer<vblex><pri><p3><pl>,3,3,5,5


In [4]:
df.drop_duplicates(inplace=True)

In [5]:
df["user_id"].str.contains(r"^u:", regex=True).all()

True

In [6]:
df["user_id"] = df["user_id"].str.slice(start=2)

In [7]:
df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s")

In [8]:
df.sort_values(by="timestamp", inplace=True)

In [9]:
LOWER_BOUND_P = 0.001  # 0.1% minimum recall probability
UPPER_BOUND_P = 0.999  # 99.9% maximum recall probability

In [10]:
df["p_recall"] = np.clip(df["p_recall"], a_min=LOWER_BOUND_P, a_max=UPPER_BOUND_P)

## Training the Model on the full data set

## Training for Part of Speech

In [114]:
HLR_WEIGHTS_PATH = "hlr_pos_model_atm5.pth"
POS_ENCODER_PATH = "pos_encoder_atm5.pkl"
EPOCHS = 5
BATCH_SIZE = 8192

In [115]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Initializing POS Neural HLR on: {device}")

Initializing POS Neural HLR on: cuda


In [116]:
(df["history_seen"] == 0).any()

np.False_

In [117]:
df["history_seen"] = df["history_seen"].astype(np.float32)
df["history_correct"] = df["history_correct"].astype(np.float32)
df["history_accuracy"] = (df["history_correct"] / (df["history_seen"])).astype(
    np.float32
)
df["toxicity"]= (df["history_seen"] / (df["history_accuracy"]*df["history_accuracy"])).astype(
    np.float32
)
df["delta_days"] = (df["delta"] / (60 * 60 * 24)).astype(np.float32)
df["p_recall"] = df["p_recall"].astype(np.float32)

In [118]:
df["pos"] = (
    df["lexeme_string"].str.extract(r"<([^>]+)>", expand=False).fillna("unknown")
)

In [119]:
print(df["pos"].value_counts())

pos
n                    5206243
vblex                2136159
det                  1346658
prn                  1046786
adj                   659528
                      ...   
@cnj:avant_que            13
@cnj:autant_que           10
@adv:a_posteriori          6
@adv:a_priori              6
ord                        4
Name: count, Length: 81, dtype: int64[pyarrow]


In [120]:
df["pos"] = df["pos"].str.replace(r"^@([^:]+):.*", r"\1", regex=True)

In [121]:
threshold = 5000

In [122]:
df['tag_cnt'] = df['lexeme_string'].str.count(r'<[^>]+>')
tag_counts=df["pos"].value_counts()
rare_tags = tag_counts[tag_counts < threshold].index
df["pos"] = df["pos"].replace(rare_tags, "other")

In [123]:
df.head(5)

,p_recall,timestamp,delta,user_id,learning_language,ui_language,lexeme_id,lexeme_string,history_seen,history_correct,session_seen,session_correct,history_accuracy,delta_days,pos,tag_cnt,toxicity
0,0.999,2013-02-28 18:28:01,27649635,FO,de,en,76390c1350a8dac31186187e2fe1e178,lernt/lernen<vblex><pri><p3><sg>,6.0,4.0,2,2,0.666667,320.018921,vblex,4,13.499999
1,0.500,2013-02-28 18:28:01,27649635,FO,de,en,7dfd7086f3671685e2cf1c1da72796d7,die/die<det><def><f><sg><nom>,4.0,4.0,2,1,1.000000,320.018921,det,5,4.000000
2,0.999,2013-02-28 18:28:01,27649635,FO,de,en,35a54c25a2cda8127343f6a82e6f6b7d,mann/mann<n><m><sg><nom>,5.0,4.0,1,1,0.800000,320.018921,n,4,7.812500
3,0.500,2013-02-28 18:28:01,27649635,FO,de,en,0cf63ffe3dda158bc3dbd55682b355ae,frau/frau<n><f><sg><nom>,6.0,5.0,2,1,0.833333,320.018921,n,4,8.640000
4,0.999,2013-02-28 18:28:01,27649635,FO,de,en,84920990d78044db53c1b012f5bf9ab5,das/das<det><def><nt><sg><nom>,4.0,4.0,1,1,1.000000,320.018921,det,5,4.000000


In [124]:
print(df["pos"].value_counts())

pos
n                 5206243
vblex             2136159
det               1346658
prn               1046811
adj                659528
vbser              545396
adv                462581
*sf                431403
pr                 399701
ij                 156753
cnjcoo             156310
cnjadv              45630
vbmod               41863
vbhaver             40578
num                 35311
vbdo                34466
cnjsub              21835
common_phrases      18071
preadv              14645
rel                 14584
other               14300
apos                 9629
vaux                 8027
gen                  7663
Name: count, dtype: int64[pyarrow]


In [125]:
split_idx = int(len(df) * 0.8)
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

In [126]:
if os.path.exists(POS_ENCODER_PATH):
    print("Loading cached POS Encoder...")
    pos_encoder = joblib.load(POS_ENCODER_PATH)
else:
    print("Fitting new POS Encoder...")
    pos_encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    pos_encoder.fit(train_df[["pos"]])
    joblib.dump(pos_encoder, POS_ENCODER_PATH)

Fitting new POS Encoder...


In [127]:
train_df["pos_idx"] = pos_encoder.transform(train_df[["pos"]])
test_df["pos_idx"] = pos_encoder.transform(test_df[["pos"]])

In [128]:
num_pos = len(pos_encoder.categories_[0])

In [129]:
# pytorch NN models hate negative numbers, so we are replacing -1s with 24
train_df["pos_idx"] = train_df["pos_idx"].replace(-1, num_pos)
test_df["pos_idx"] = test_df["pos_idx"].replace(-1, num_pos)

In [130]:
cont_features = ["history_seen", "history_correct", "tag_cnt","history_accuracy","toxicity"]

In [131]:
def create_tensors(
    data_subset: pd.DataFrame,
) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Converts specific columns from a spaced-repetition pandas DataFrame into PyTorch tensors.

    This function extracts categorical part-of-speech indices, continuous historical
    performance features, time elapsed (delta), and target recall probabilities, formatting
    them with the specific data types and matrix shapes required by the PyTorch model.

    Args:
        data_subset (pd.DataFrame): A chunk of the dataset. Must contain the columns
            'pos_idx', 'delta_days', 'p_recall', and the columns listed in the global
            `cont_features` list.

    Returns:
        Tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor]:
            - p_idx: 1D tensor of categorical part-of-speech indices (dtype: torch.long).
            - cont: 2D tensor of continuous historical features (dtype: torch.float32).
            - deltas: 2D column vector of time elapsed since last review (dtype: torch.float32, shape: [N, 1]).
            - targets: 2D column vector of target recall probabilities (dtype: torch.float32, shape: [N, 1]).
    """

    p_idx = torch.tensor(
        data_subset["pos_idx"].values.astype(np.int64), dtype=torch.long
    )
    cont = torch.tensor(
        data_subset[cont_features].values.astype(np.float32), dtype=torch.float32
    )
    deltas = torch.tensor(
        data_subset["delta_days"].values.astype(np.float32), dtype=torch.float32
    ).unsqueeze(1)
    targets = torch.tensor(
        data_subset["p_recall"].values.astype(np.float32), dtype=torch.float32
    ).unsqueeze(1)
    return p_idx, cont, deltas, targets

In [132]:
train_p, train_c, train_d, train_y = create_tensors(train_df)
test_p, test_c, test_d, test_y = create_tensors(test_df)

In [133]:
train_dataset = TensorDataset(train_p, train_c, train_d, train_y)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [134]:
class PosHalfLifeRegression(nn.Module):
    """
    A neural network model for Half-Life Regression (HLR) using Part-of-Speech tags.

    This model predicts the probability of a student recalling a specific word/concept.
    It estimates the memory "half-life" (h) using a categorical Part-of-Speech embedding
    concatenated with historical continuous features. It then applies the Ebbinghaus
    forgetting curve formula: $p = 2^{-\\frac{\\Delta}{h}}$.

    Attributes:
        pos_emb (nn.Embedding): Embedding layer for the categorical POS tags.
        fc1 (nn.Linear): First fully connected hidden layer.
        fc2 (nn.Linear): Second fully connected hidden layer.
        fc_out (nn.Linear): Final linear layer to output the raw half-life estimate.
        relu (nn.ReLU): ReLU activation function for hidden layers.
        softplus (nn.Softplus): Softplus activation to ensure the half-life is positive.
    """

    def __init__(self, num_pos, pos_emb_dim=4):
        """
        Initializes the POS Half-Life Regression model.

        Args:
            num_pos (int): The total number of known Part-of-Speech categories.
                The embedding layer will automatically add +1 to accommodate the
                "unknown" (-1) category index.
            pos_emb_dim (int, optional): The dimensionality of the POS embedding vector.
                Defaults to 4.
        """

        super(PosHalfLifeRegression, self).__init__()

        self.pos_emb = nn.Embedding(num_pos + 1, pos_emb_dim)

        # Input = POS(4) + Cont(3) = 7
        self.fc1 = nn.Linear(pos_emb_dim + 5, 32)
        self.fc2 = nn.Linear(32, 16)
        self.fc_out = nn.Linear(16, 1)

        # Gradient Death Fix 1: Bias head start
        nn.init.constant_(self.fc_out.bias, 2.5)

        self.relu = nn.ReLU()
        self.softplus = nn.Softplus()

    def forward(self, pos_idx, cont_features, delta_days):
        """
        Executes the forward pass of the network to calculate recall probability.

        Args:
            pos_idx (torch.Tensor): 1D tensor of integer POS indices.
                Shape: [batch_size]
            cont_features (torch.Tensor): 2D tensor of continuous historical features
                (e.g., seen, correct, accuracy). Shape: [batch_size, 3]
            delta_days (torch.Tensor): 2D tensor of days elapsed since the item was
                last seen. Shape: [batch_size, 1]

        Returns:
            torch.Tensor: The predicted probability of recall (between 0.0 and 1.0).
                Shape: [batch_size, 1]
        """

        p = self.pos_emb(pos_idx)

        # Combine POS embedding with the 3 continuous features
        x = torch.cat([p, cont_features], dim=1)

        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))

        h = self.softplus(self.fc_out(x)) + 1e-3

        # Gradient Death Fix 2: Clamp the exponent
        exponent = torch.clamp(delta_days / h, min=0.0, max=20.0)
        p_recall = torch.pow(2.0, -exponent)

        return p_recall

    def get_halflife(
        self, pos_idx: torch.Tensor, cont_features: torch.Tensor
    ) -> torch.Tensor:
        """
        Calculates the memory half-life (in days) directly, bypassing the decay curve.
        Notice that delta_days is NOT required for this calculation!
        """

        p = self.pos_emb(pos_idx)
        x = torch.cat([p, cont_features], dim=1)

        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))

        h = self.softplus(self.fc_out(x)) + 1e-3

        return h

In [135]:
hlr_model = PosHalfLifeRegression(num_pos).to(device)
criterion = nn.L1Loss()
optimizer = optim.Adam(hlr_model.parameters(), lr=0.005)

In [136]:
if os.path.exists(HLR_WEIGHTS_PATH):
    print(f"\n[CACHE HIT] Loading pre-trained weights...")
    hlr_model.load_state_dict(
        torch.load(HLR_WEIGHTS_PATH, weights_only=True, map_location=device)
    )
else:
    print(f"\n[TRAINING] Starting fresh training...")
    for epoch in range(EPOCHS):
        hlr_model.train()
        total_loss = 0

        
        
        # Back to 4 variables!
    for batch_p, batch_c, batch_d, batch_y in train_loader:
    
    # Send all 4 to the GPU
        batch_p = batch_p.to(device)
        batch_c, batch_d, batch_y = (
            batch_c.to(device),
            batch_d.to(device),
            batch_y.to(device),
    )

        optimizer.zero_grad()
    
    # Pass the 3 inputs to your model
        predictions = hlr_model(batch_p, batch_c, batch_d)

    # Calculate loss against your single probability target
        loss = criterion(predictions, batch_y)
    
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_p.size(0)


        avg_train_loss = total_loss / len(train_p)
        print(f"   Epoch {epoch+1}/{EPOCHS} | Train MAE: {avg_train_loss:.4f}")

    print("Saving model weights to disk...")
    torch.save(hlr_model.state_dict(), HLR_WEIGHTS_PATH)




[TRAINING] Starting fresh training...
   Epoch 5/5 | Train MAE: 0.0003
   Epoch 5/5 | Train MAE: 0.0006
   Epoch 5/5 | Train MAE: 0.0008
   Epoch 5/5 | Train MAE: 0.0011
   Epoch 5/5 | Train MAE: 0.0013
   Epoch 5/5 | Train MAE: 0.0016
   Epoch 5/5 | Train MAE: 0.0018
   Epoch 5/5 | Train MAE: 0.0021
   Epoch 5/5 | Train MAE: 0.0023
   Epoch 5/5 | Train MAE: 0.0026
   Epoch 5/5 | Train MAE: 0.0028
   Epoch 5/5 | Train MAE: 0.0030
   Epoch 5/5 | Train MAE: 0.0032
   Epoch 5/5 | Train MAE: 0.0034
   Epoch 5/5 | Train MAE: 0.0036
   Epoch 5/5 | Train MAE: 0.0038
   Epoch 5/5 | Train MAE: 0.0040
   Epoch 5/5 | Train MAE: 0.0042
   Epoch 5/5 | Train MAE: 0.0044
   Epoch 5/5 | Train MAE: 0.0046
   Epoch 5/5 | Train MAE: 0.0048
   Epoch 5/5 | Train MAE: 0.0049
   Epoch 5/5 | Train MAE: 0.0051
   Epoch 5/5 | Train MAE: 0.0053
   Epoch 5/5 | Train MAE: 0.0054
   Epoch 5/5 | Train MAE: 0.0056
   Epoch 5/5 | Train MAE: 0.0057
   Epoch 5/5 | Train MAE: 0.0059
   Epoch 5/5 | Train MAE: 0.0060
   E

In [137]:
from sklearn.metrics import mean_absolute_error

print("\nEvaluating Model on test set...")
hlr_model.eval()

with torch.no_grad():
    # 1. Move test data to the device
    test_p = test_p.to(device)
    test_c, test_d = test_c.to(device), test_d.to(device)

    # 2. FIX: No tuple unpacking! Just catch the single prediction tensor
    test_preds = hlr_model(test_p, test_c, test_d)
    
    # 3. Move predictions to CPU and convert to NumPy
    test_preds_np = test_preds.cpu().numpy()
    
    # 4. Make sure test_y is also on the CPU
    test_y_np = test_y.cpu().numpy()

# Calculate MAE
mae_hlr = mean_absolute_error(test_y_np, test_preds_np)
print(f"Final Test MAE: {mae_hlr:.4f}")


Evaluating Model on test set...
Final Test MAE: 0.1053


In [138]:
hlr_model.eval()


with torch.no_grad():
    test_p = test_p.to(device)
    test_c = test_c.to(device)

    nn_halflife_pos = hlr_model.get_halflife(test_p, test_c)

    nn_halflife_pos_np = nn_halflife_pos.squeeze().cpu().numpy()

pos_results_df = pd.DataFrame(
    {
        "delta_days": test_df["delta_days"].values,
        "actual_p_recall": test_y_np.squeeze(),
        "predicted_p_recall": test_preds_np.squeeze(),
        "nn_predicted_halflife": nn_halflife_pos_np,
    }
)

print(pos_results_df.sample(10))

         delta_days  actual_p_recall  predicted_p_recall  \
1348025    0.834097            0.999            0.999964   
1311064    0.002303            0.999            1.000000   
1266518    5.988739            0.999            0.999291   
2144504    0.510440            0.999            0.999953   
2041979  118.400276            0.001            0.979711   
1492972    0.004792            0.999            0.999999   
2454502   35.707420            0.999            0.995525   
2520596    0.002083            0.999            0.999999   
1589013    0.416192            0.999            0.999996   
2251336    1.541204            0.999            0.999685   

         nn_predicted_halflife  
1348025           16211.931641  
1311064           11420.798828  
1266518            5851.933105  
2144504            7527.405762  
2041979            4003.727295  
1492972            6025.975098  
2454502            5518.937500  
2520596            2035.403198  
1589013           69427.343750  
2251336  

In [139]:
(pos_results_df["nn_predicted_halflife"] / (60 * 60 * 24)).mean()

np.float32(0.10490693)

In [140]:
pos_results_df["predicted_p_recall"].mean()

np.float32(0.9983711)

In [141]:
def prep_duolingo_data(df: pd.DataFrame) -> pd.DataFrame:
    """Applies the exact mathematical transformations from the Settles & Meeder paper."""
    df = df.copy()
    
    # 1. Square root transformation for historical counts
    df['right_transformed'] = np.sqrt(1 + df['history_correct'])
    df['wrong_transformed'] = np.sqrt(1 + (df['history_seen'] - df['history_correct']))
    
    # 2. Calculate the "Target" Half-Life for training the dual-loss function
    # Paper bounds: min 15 mins, max 274 days
    p_safe = np.clip(df['p_recall'], 0.0001, 0.9999)
    h_target = -df['delta_days'] / np.log2(p_safe)
    df['h_target'] = np.clip(h_target, 15.0 / (24 * 60), 274.0)
    
    return df

print("🧪 Transforming data to match original paper...")
train_df_prep = prep_duolingo_data(train_df)
test_df_prep = prep_duolingo_data(test_df)

🧪 Transforming data to match original paper...


In [142]:
train_df_prep.head()

,p_recall,timestamp,delta,user_id,learning_language,ui_language,lexeme_id,lexeme_string,history_seen,history_correct,...,session_correct,history_accuracy,delta_days,pos,tag_cnt,toxicity,pos_idx,right_transformed,wrong_transformed,h_target
0,0.999,2013-02-28 18:28:01,27649635,FO,de,en,76390c1350a8dac31186187e2fe1e178,lernt/lernen<vblex><pri><p3><sg>,6.0,4.0,...,2,0.666667,320.018921,vblex,4,13.499999,21.0,2.236068,1.732051,274.0
1,0.500,2013-02-28 18:28:01,27649635,FO,de,en,7dfd7086f3671685e2cf1c1da72796d7,die/die<det><def><f><sg><nom>,4.0,4.0,...,1,1.000000,320.018921,det,5,4.000000,8.0,2.236068,1.000000,274.0
2,0.999,2013-02-28 18:28:01,27649635,FO,de,en,35a54c25a2cda8127343f6a82e6f6b7d,mann/mann<n><m><sg><nom>,5.0,4.0,...,1,0.800000,320.018921,n,4,7.812500,11.0,2.236068,1.414214,274.0
3,0.500,2013-02-28 18:28:01,27649635,FO,de,en,0cf63ffe3dda158bc3dbd55682b355ae,frau/frau<n><f><sg><nom>,6.0,5.0,...,1,0.833333,320.018921,n,4,8.640000,11.0,2.449490,1.414214,274.0
4,0.999,2013-02-28 18:28:01,27649635,FO,de,en,84920990d78044db53c1b012f5bf9ab5,das/das<det><def><nt><sg><nom>,4.0,4.0,...,1,1.000000,320.018921,det,5,4.000000,8.0,2.236068,1.000000,274.0


In [156]:
cont_features = ['right_transformed', 'wrong_transformed']

def create_original_tensors(data_subset: pd.DataFrame, cont_feats: list) -> Tuple[torch.Tensor, ...]:
    pos = torch.tensor(data_subset["pos_idx"].values.astype(np.int64), dtype=torch.long)
    cont = torch.tensor(data_subset[cont_feats].values.astype(np.float32), dtype=torch.float32)
    deltas = torch.tensor(data_subset["delta_days"].values.astype(np.float32), dtype=torch.float32).unsqueeze(1)
    
    # We now need TWO targets: Probability and Half-Life
    p_targets = torch.tensor(data_subset["p_recall"].values.astype(np.float32), dtype=torch.float32).unsqueeze(1)
    h_targets = torch.tensor(data_subset["h_target"].values.astype(np.float32), dtype=torch.float32).unsqueeze(1)
    
    return pos, cont, deltas, p_targets, h_targets

print("📦 Generating tensors...")
train_pos, train_c, train_d, train_y_p, train_y_h = create_original_tensors(train_df_prep, cont_features)
test_pos, test_c, test_d, test_y_p, test_y_h = create_original_tensors(test_df_prep, cont_features)

📦 Generating tensors...


In [157]:
BATCH_SIZE = 8096
train_dataset = TensorDataset(train_pos, train_c, train_d, train_y_p, train_y_h)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
MODEL_CACHE_PATH = "original_hlr_pos.pth"

In [161]:
class OriginalHLRWithPOS(nn.Module):
    def __init__(self, num_cont_features: int, num_pos_tags: int) -> None:
        super(OriginalHLRWithPOS, self).__init__()
        # Continuous weights
        self.linear = nn.Linear(num_cont_features, 1)
        # Categorical weights (replaces the 1.0 dummy variable dictionary)
        self.pos_weight = nn.Embedding(num_pos_tags, 1)
        # Bias start
        nn.init.constant_(self.linear.bias, 2.5) 

    def forward(self, pos_idx: torch.Tensor, cont_features: torch.Tensor, delta_days: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        # Dot product of features + weights
        dp = self.linear(cont_features) + self.pos_weight(pos_idx)
        
        # Calculate h (with the paper's max clipping)
        dp = torch.clamp(dp, max=15.0) 
        h = torch.pow(2.0, dp)

        # Calculate p
        exponent = torch.clamp(delta_days / h, min=0.0, max=20.0)
        p_recall = torch.pow(2.0, -exponent)

        # Return BOTH so we can calculate the dual-loss
        return p_recall, h
        
    def get_halflife(self, pos_idx: torch.Tensor, cont_features: torch.Tensor) -> torch.Tensor:
        dp = self.linear(cont_features) + self.pos_weight(pos_idx)
        dp = torch.clamp(dp, max=15.0)
        return torch.pow(2.0, dp)

In [162]:
model = OriginalHLRWithPOS(num_cont_features=len(cont_features), num_pos_tags=num_pos).to(device)

if os.path.exists(MODEL_CACHE_PATH):
    print(f"📦 Found cached model! Loading weights from {MODEL_CACHE_PATH}...")
    model.load_state_dict(torch.load(MODEL_CACHE_PATH, map_location=device, weights_only=True))
else:
    print("⚙️ No cached model found. Starting Training...")
    
    # We use MSE to match the paper's slp (squared loss p) and slh (squared loss h)
    criterion = nn.MSELoss() 
    
    # Adam optimizer with weight_decay handles the paper's L2 Regularization (l2wt)
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.1)
    
    # Paper's Half-Life weight multiplier
    HL_WT = 0.01 
    EPOCHS = 5

    print("🚀 Optimizing for both P and H...")
    for epoch in range(EPOCHS):
        model.train() 
        total_loss = 0.0
        
        for b_pos, b_c, b_d, b_y_p, b_y_h in train_loader:
            b_pos, b_c, b_d = b_pos.to(device), b_c.to(device), b_d.to(device)
            b_y_p, b_y_h = b_y_p.to(device), b_y_h.to(device)
            
            optimizer.zero_grad()
            
            # Get predictions
            pred_p, pred_h = model(b_pos, b_c, b_d)
            
            # Calculate Squared Loss for P and H
            loss_p = criterion(pred_p, b_y_p)
            loss_h = criterion(pred_h, b_y_h)
            
            # Combine exactly as paper describes: slp + hlwt*slh
            loss = loss_p + (HL_WT * loss_h)
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            
        print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss (Combined MSE): {total_loss / len(train_loader):.4f}")

    print("✅ Training Complete!")
    
    # Save the trained weights for next time
    torch.save(model.state_dict(), MODEL_CACHE_PATH)
    print(f"💾 Model weights saved to {MODEL_CACHE_PATH}\n")

📦 Found cached model! Loading weights from original_hlr_pos.pth...


In [163]:
print("📊 Evaluating Model on Test Set...")

model.eval()
with torch.no_grad():
    test_pos = test_pos.to(device)
    test_c, test_d = test_c.to(device), test_d.to(device)

    # We only need the p_recall prediction for the final MAE score
    test_preds_p, _ = model(test_pos, test_c, test_d)
    
    test_preds_np = test_preds_p.cpu().numpy()
    test_y_np = test_y_p.cpu().numpy()

# Calculate the MAE of the probabilities
mae_final = mean_absolute_error(test_y_np, test_preds_np)
print(f"Final Test MAE: {mae_final:.4f}")

📊 Evaluating Model on Test Set...
Final Test MAE: 0.1294


In [110]:
num_pos

24